In [ ]:
import irw
import numpy as np
import pandas as pd

In [ ]:
gilbert = irw.fetch("gilbert_meta_2")

In [ ]:
response_dataframe = gilbert.pivot(index = "id", columns = "item", values = "resp")
x_sq = response_dataframe.to_numpy(dtype = float)
no_items, no_students = gilbert["item"].nunique(), gilbert["id"].nunique()

def sigma(z):
    return 1 / (1 + np.exp(-z))

def p_correct(a_s,d_q):
    return sigma(a_s - d_q)

A_s = x_sq.sum(axis = 1)
D_q = x_sq.sum(axis = 0)

In [ ]:
def log_likelihood(x_sq, a_s, d_q):

    a_minus_d = a_s[:,None] - d_q[None,:]
    answer = (((x_sq)*(a_minus_d)) - (np.logaddexp(0,a_minus_d))).sum()

    return answer

def maximising_f(f, xmin, xmax, maxiter = 50, tol = 1e-8):

    a = xmin
    b = xmax

    for _ in range(maxiter):

        m = (a + b)/2
        l = (a + m)/2
        r = (b + m)/2

        vals = np.array([f(a), f(b), f(m), f(l), f(r)])

        max_val = np.argmax(vals)

        if max_val == 0 or max_val == 3:
            b = m 
        elif max_val == 1 or max_val == 4:
            a = m
        else:
            a = l
            b = r 
        
        if abs(a - b) < tol:
            break
        
        
    return (a + b)/2

In [ ]:
# gradient 'ascent' code
a_s = np.zeros(no_students)
d_q = np.zeros(no_items)
gamma_a = None
gamma_d = None
gamma_converged = False 

tol = 1e-5
gamma_a_tol = 1e-2
gamma_d_tol = 1e-3
max_iterations = 10000

log_likelihoods = []

for i in range(max_iterations):

    P_sq = sigma(a_s[:,None] - d_q[None,:])

    expected_A_s = P_sq.sum(axis = 1)
    expected_D_q = P_sq.sum(axis = 0)

    dl_da = A_s - expected_A_s
    dl_dd = expected_D_q - D_q

    if not gamma_converged:
        likelihood_a = lambda n: log_likelihood(x_sq, a_s + (n*dl_da), d_q)
        likelihood_d = lambda n: log_likelihood(x_sq, a_s, d_q + (n*dl_dd))

        new_gamma_a = maximising_f(likelihood_a, 0, 0.5)
        new_gamma_d = maximising_f(likelihood_d, 0, 0.5)

        if gamma_a is not None and gamma_d is not None:

            if abs(new_gamma_a - gamma_a) < gamma_a_tol and abs(new_gamma_d - gamma_d) < gamma_d_tol:
                gamma_converged = True
                print(f"iteration {i}, Both gammas converged")
                print(f"Gamma a = {new_gamma_a}, Gamma b = {new_gamma_d}")

    gamma_a = new_gamma_a
    gamma_d = new_gamma_d
    
    a_s += gamma_a * dl_da
    d_q += gamma_d * dl_dd
    
    mean_d = d_q.mean()
    d_q -= mean_d
    a_s -= mean_d
        
    value = log_likelihood(x_sq, a_s, d_q)
    log_likelihoods.append(value)

    if i != 0 and abs(value - log_likelihoods[-2]) < tol:
        final_iteration = i
        print()
        print("Converged!")
        print("Convergence details:")
        print(f"iteration: {final_iteration}")
        print(f"max likelihood value: {value:.2f}")
        break


In [ ]:
print(a_s)
print(d_q)

In [ ]:
class_intervals = [
    (1, 5),
    (6, 7),
    (8, 8),
    (9, 9),
    (10, 11),
    (12, 12),
    (13, 13),
    (14, 14),
    (15, 16),
    (17, 19)
]

# learnt about boolean masks here
filtered_A = A_s[(A_s > 0) & (A_s < no_items)]
filtered_a = a_s[(A_s > 0) & (A_s < no_items)]
filtered_x = x_sq[(A_s > 0) & (A_s < no_items)]

print(filtered_a.size)
print(filtered_A.size)
print(filtered_x.shape)

In [ ]:
for lower_score, upper_score in class_intervals:

    boolean_mask = (filtered_A >= lower_score) & (filtered_A <= upper_score)
    N_g = boolean_mask.sum()
    mean_a_g = filtered_a[boolean_mask].mean()

    class_responses = filtered_x[boolean_mask]
    T_gi = class_responses.sum(axis = 0)

    observed_p_gi = T_gi / N_g
    expected_p_gi = sigma(mean_a_g - d_q)


    print(f"g = total scores {lower_score}-{upper_score}: N = {N_g}, mean ability = {mean_a_g:.2f}")
    print(f"Number correct for each item: {T_gi}")
    print(f"Observed proportion correct: {observed_p_gi.round(3)}")
    print(f"Expected proportion correct: {expected_p_gi.round(3)}")

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
def generate_one_ICC(item):

    item_index = item - 1
    class_abilities = []
    observed_proportions = []

    for lower_score, upper_score in class_intervals:

        boolean_mask = (filtered_A >= lower_score) & (filtered_A <= upper_score)

        N_g = boolean_mask.sum()
        mean_a_g = filtered_a[boolean_mask].mean()

        class_responses = filtered_x[boolean_mask]
        T_gi = class_responses.sum(axis=0)

        observed_p_gi = T_gi / N_g

        class_abilities.append(mean_a_g)
        observed_proportions.append(observed_p_gi[item_index])

    item_difficulty = d_q[item_index]
    ability_values = np.linspace(min(filtered_a), max(filtered_a), 100)
    expected_probabilities = sigma(ability_values - item_difficulty)

    plt.plot(ability_values, expected_probabilities, label = "Rasch model ICC")
    plt.scatter(class_abilities, observed_proportions, label = "Observed values for different class intervals")

    plt.title(f"Item {item} ICC")
    plt.xlabel("Person ability")
    plt.ylabel("Proportion Correct")
    
    plt.grid()
    plt.legend()
    plt.show()

In [ ]:
generate_one_ICC(1)

In [ ]:
def generate_ICC(item,ax):

    item_index = item - 1
    class_abilities = []
    observed_proportions = []

    for lower_score, upper_score in class_intervals:

        boolean_mask = (filtered_A >= lower_score) & (filtered_A <= upper_score)

        N_g = boolean_mask.sum()
        mean_a_g = filtered_a[boolean_mask].mean()

        class_responses = filtered_x[boolean_mask]
        T_gi = class_responses.sum(axis=0)

        observed_p_gi = T_gi / N_g

        class_abilities.append(mean_a_g)
        observed_proportions.append(observed_p_gi[item_index])

    item_difficulty = d_q[item_index]
    ability_values = np.linspace(min(filtered_a), max(filtered_a), 100)
    expected_probabilities = sigma(ability_values - item_difficulty)

    ax.plot(ability_values, expected_probabilities)
    ax.scatter(class_abilities, observed_proportions)

    ax.set_title(f"Item {item} ICC")
    ax.set_xlabel("Person ability")
    ax.set_ylabel("Proportion Correct")

    ax.grid(True)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 8))
for item in range(1, 11):
    generate_ICC(item, axes.flat[item - 1])

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 8))

for item in range(11, 21):
    generate_ICC(item, axes.flat[item - 11])

plt.tight_layout()
plt.show()

In [ ]:
chi_squared = 0

for lower_score, upper_score in class_intervals:

        boolean_mask = (filtered_A >= lower_score) & (filtered_A <= upper_score)

        N_g = boolean_mask.sum()
        mean_a_g = filtered_a[boolean_mask].mean()

        class_responses = filtered_x[boolean_mask]
        T_gi = class_responses.sum(axis=0)
        P_gi = sigma(mean_a_g - d_q[0])
        E_gi = N_g * P_gi

        Z_gi_squared = ((T_gi[0] - E_gi) / np.sqrt(N_g * P_gi * (1 - P_gi)))**2

        chi_squared += Z_gi_squared

print(chi_squared)

In [ ]:
def calculate_chi_squared(item):
    item_index = item - 1
    chi_squared = 0

    for lower_score, upper_score in class_intervals:

        boolean_mask = (filtered_A >= lower_score) & (filtered_A <= upper_score)

        N_g = boolean_mask.sum()
        mean_a_g = filtered_a[boolean_mask].mean()

        class_responses = filtered_x[boolean_mask]
        T_gi = class_responses.sum(axis=0)
        P_gi = sigma(mean_a_g - d_q[item_index])
        E_gi = N_g * P_gi

        Z_gi_squared = ((T_gi[item_index] - E_gi) / np.sqrt(N_g * P_gi * (1 - P_gi)))**2

        chi_squared += Z_gi_squared

    return chi_squared

In [ ]:
calculate_chi_squared(1)

In [ ]:
chi_squared_values = []

for i in range(1,21):
    chi_squared_values.append(calculate_chi_squared(i))


In [ ]:
for i in range(1,21):     
    print(f"item {i}: {calculate_chi_squared(i)}")

In [ ]:
# chi squared critical value for 9 DOF and 0.05 sig is 16.919